# CBMS — Activity Classifier Training

Trains a 2-layer LSTM on MediaPipe Pose keypoint sequences.

**Dataset layout expected:**
```
/kaggle/input/cbms-training-data/
    spitting/    ← short video clips of spitting gesture
    normal/      ← clips of walking, talking, yawning, drinking
    littering/   ← (optional) add more folders to add more classes
```

> After Cell 1: restart session, then run from Cell 2.

## Cell 1 — Install

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "mediapipe==0.10.14", "opencv-python-headless"])
print("Done. Restart session now, then run from Cell 2.")

## Cell 2 — Configuration

In [8]:
import os, json, random, numpy as np
from pathlib import Path

DATA_DIR    = "/kaggle/input/datasets/gallimus/cbms-training-data"
OUTPUT_DIR  = "/kaggle/working/"
MODEL_PATH  = os.path.join(OUTPUT_DIR, "activity_classifier.pt")
LABELS_PATH = os.path.join(OUTPUT_DIR, "class_labels.json")

# Sequence settings
N_FRAMES    = 32      # Updated to 32 frames for better context
N_KEYPOINTS = 33      
FEATURES    = N_KEYPOINTS * 3   # 99 features

# Model architecture - UPDATED
HIDDEN_1    = 256
HIDDEN_2    = 128
DROPOUT     = 0.3

# Training
EPOCHS      = 60
BATCH_SIZE  = 16
LR          = 1e-3
VAL_SPLIT   = 0.20

# Augmentation
AUGMENT     = True

# Reproducibility
SEED        = 42
random.seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Config loaded.")


Config loaded.


## Cell 3 — Extract pose sequences from video clips

In [3]:
import cv2
import mediapipe as mp

mp_pose = mp.solutions.pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.3,
)

def extract_sequences(video_path: str, n_frames: int = N_FRAMES, overlap: int = 16) -> list:
    """
    The 'Big Fix': Sliding Window + Zero-Centering (Nose-relative).
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    all_frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = mp_pose.process(rgb)

        if result.pose_landmarks:
            kp = result.pose_landmarks.landmark
            nose = kp[0]
            # Zero-Centering: Translation Invariant
            vec = np.array(
                [[lm.x - nose.x, lm.y - nose.y, lm.z] for lm in kp],
                dtype=np.float32
            ).flatten()
        else:
            vec = np.zeros(FEATURES, dtype=np.float32)
        
        all_frames.append(vec)
    cap.release()

    if len(all_frames) < n_frames: return []
        
    chunks = []
    step = n_frames - overlap
    for start in range(0, len(all_frames) - n_frames + 1, step):
        chunks.append(np.stack(all_frames[start : start + n_frames]))
    return chunks

def load_dataset(data_dir: str):
    data_path = Path(data_dir)
    class_names = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
    
    sequences, labels = [], []
    for class_idx, name in enumerate(class_names):
        clip_files = list((data_path / name).glob("*.mp4")) + list((data_path / name).glob("*.avi"))
        ok = 0
        for clip_path in clip_files:
            chunks = extract_sequences(str(clip_path))
            for seq in chunks:
                sequences.append(seq)
                labels.append(class_idx)
            if chunks: ok += 1
        print(f"  {name:<15} {ok:>3} clips -> {len(sequences)} chunks")
    return np.stack(sequences), np.array(labels), class_names

X, y, CLASS_NAMES = load_dataset(DATA_DIR)


W0000 00:00:1775994245.436527     415 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775994245.470056     416 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  littering        74 clips -> 314 chunks
  normal          161 clips -> 1455 chunks
  spitting         38 clips -> 1857 chunks


## Cell 4 — Augmentation & train/val split

In [5]:
def augment_sequence(seq: np.ndarray, training=True) -> np.ndarray:
    """
    Mirror flip + Noise + Temporal Jitter.
    """
    n, f = seq.shape
    aug = seq.copy().reshape(n, N_KEYPOINTS, 3)
    
    # Mirror swap (x = -x)
    if random.random() > 0.5:
        aug[:, :, 0] = -aug[:, :, 0]

    # Temporal Jitter: Simulate flicker by skipping frames
    if training and random.random() > 0.5:
        skip = random.randint(1, 2)
        idx = sorted(random.sample(range(n), n - skip))
        aug = np.concatenate([aug[idx], np.tile(aug[idx[-1]], (skip, 1, 1))])
    
    # Gaussian noise
    aug += np.random.normal(0, 0.005, aug.shape).astype(np.float32)
    return aug.reshape(n, f)

if AUGMENT:
    aug_seqs, aug_labels = [], []
    for seq, label in zip(X, y):
        aug_seqs.append(augment_sequence(seq))
        aug_labels.append(label)
        if CLASS_NAMES[label] != "normal":
            for _ in range(2):
                aug_seqs.append(augment_sequence(seq))
                aug_labels.append(label)
    X = np.concatenate([X, np.stack(aug_seqs)])
    y = np.concatenate([y, np.array(aug_labels)])

idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]
split = int(len(X) * (1 - VAL_SPLIT))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]


## Cell 5 — Model definition

In [9]:
import torch
import torch.nn as nn

class PoseActivityClassifier(nn.Module):
    def __init__(self, input_size, hidden1, hidden2, num_classes, dropout):
        super().__init__()
        self.gru1 = nn.GRU(input_size, hidden1, batch_first=True, bidirectional=True)
        self.gru2 = nn.GRU(hidden1*2, hidden2, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden2*2, num_classes)

    def forward(self, x):
        out, _ = self.gru1(x)
        out, _ = self.gru2(out)
        # Mean pooling across time dimension
        out = torch.mean(out, dim=1)
        return self.classifier(self.dropout(out))

model = PoseActivityClassifier(FEATURES, HIDDEN_1, HIDDEN_2, len(CLASS_NAMES), DROPOUT).to(DEVICE)


## Cell 6 — Training loop

In [10]:
from torch.utils.data import DataLoader, TensorDataset

# Convert to tensors
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_vl = torch.tensor(X_val,   dtype=torch.float32)
y_vl = torch.tensor(y_val,   dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_vl, y_vl),
    batch_size=BATCH_SIZE,
)

class_weights = torch.tensor([
    161/74,   # index 0 = littering
    161/161,  # index 1 = normal
    161/38,   # index 2 = spitting  ← now gets the highest weight
], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=8
)

best_val_acc   = 0.0
best_state     = None
history        = {"train_loss": [], "val_acc": []}

print(f"Training for {EPOCHS} epochs...")
print(f"{'Epoch':>6}  {'Train loss':>11}  {'Val acc':>8}")
print("-" * 34)

for epoch in range(1, EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)
    avg_loss = total_loss / len(X_train)

    # ── Validate ─────────────────────────────────────────────────
    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds   = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
    val_acc = correct / len(X_val)

    scheduler.step(val_acc)
    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    current_lr = optimizer.param_groups[0]["lr"]
    if epoch % 5 == 0:
        print(f"{epoch:>6}  {avg_loss:>11.4f}  {val_acc:>7.1%}  lr={current_lr:.2e}"
              + ("  ← best" if val_acc == best_val_acc else ""))

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.1%}")


Training for 60 epochs...
 Epoch   Train loss   Val acc
----------------------------------
     5       0.0868    96.1%  lr=1.00e-03  ← best
    10       0.0275    98.7%  lr=1.00e-03  ← best
    15       0.0255    99.3%  lr=1.00e-03
    20       0.0103    99.4%  lr=1.00e-03
    25       0.0120    99.7%  lr=1.00e-03
    30       0.0000   100.0%  lr=5.00e-04  ← best
    35       0.0000   100.0%  lr=5.00e-04  ← best
    40       0.0000   100.0%  lr=2.50e-04  ← best
    45       0.0000   100.0%  lr=2.50e-04  ← best
    50       0.0000   100.0%  lr=1.25e-04  ← best
    55       0.0000   100.0%  lr=6.25e-05  ← best
    60       0.0000   100.0%  lr=6.25e-05  ← best

Training complete. Best val accuracy: 100.0%


## Cell 7 — Save model & labels

In [11]:
# Load best weights back into model
model.load_state_dict(best_state)
model.eval()

# Save checkpoint
torch.save({
    "model_state": best_state,
    "class_names": CLASS_NAMES,
    "n_frames":    N_FRAMES,
    "features":    FEATURES,
    "hidden1":     HIDDEN_1,
    "hidden2":     HIDDEN_2,
    "val_accuracy": best_val_acc,
}, MODEL_PATH)

# Save class labels separately (needed by the pipeline notebook)
with open(LABELS_PATH, "w") as f:
    json.dump(CLASS_NAMES, f)

print(f"Model saved  → {MODEL_PATH}")
print(f"Labels saved → {LABELS_PATH}")
print(f"Val accuracy : {best_val_acc:.1%}")
print(f"Classes      : {CLASS_NAMES}")
print()
print(">>> Download both files from the Output tab <<<")
print(">>> Place them in backend/cv_pipeline/models/ on your local machine <<<")


Model saved  → /kaggle/working/activity_classifier.pt
Labels saved → /kaggle/working/class_labels.json
Val accuracy : 100.0%
Classes      : ['littering', 'normal', 'spitting']

>>> Download both files from the Output tab <<<
>>> Place them in backend/cv_pipeline/models/ on your local machine <<<


## Cell 8 — Evaluation & confusion matrix

In [13]:
import torch
import torch.nn.functional as F

model.eval()
all_preds, all_true = [], []

NUM_CLASSES = 3

with torch.no_grad():
    for xb, yb in val_loader:
        xb    = xb.to(DEVICE)
        probs = F.softmax(model(xb), dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

# Confusion matrix
print("Confusion matrix (rows=actual, cols=predicted):")
print(f"{'':>12}", end="")
for name in CLASS_NAMES:
    print(f"  {name[:8]:>8}", end="")
print()

for i, true_name in enumerate(CLASS_NAMES):
    print(f"  {true_name[:10]:<10}", end="")
    for j in range(NUM_CLASSES):
        count = ((all_true == i) & (all_preds == j)).sum()
        print(f"  {count:>8}", end="")
    print()

# Per-class accuracy
print()
for i, name in enumerate(CLASS_NAMES):
    mask = all_true == i
    if mask.sum() == 0:
        continue
    acc  = (all_preds[mask] == i).mean()
    print(f"  {name:<15}  accuracy: {acc:.1%}  ({mask.sum()} samples)")

# Overall
overall = (all_preds == all_true).mean()
print(f"\n  Overall accuracy: {overall:.1%}")


Confusion matrix (rows=actual, cols=predicted):
              litterin    normal  spitting
  littering       1020         0         0
  normal             0       906         0
  spitting           0         0      1278

  littering        accuracy: 100.0%  (1020 samples)
  normal           accuracy: 100.0%  (906 samples)
  spitting         accuracy: 100.0%  (1278 samples)

  Overall accuracy: 100.0%


## Cell 9 — Single clip inference test
Test the saved model on one clip before integrating it into the pipeline.

In [16]:
def predict_clip(video_path: str, model_path: str) -> dict:
    ckpt = torch.load(model_path, map_location="cpu")
    clf = PoseActivityClassifier(ckpt['features'], ckpt['hidden1'], ckpt['hidden2'], len(ckpt['class_names']), 0.0)
    clf.load_state_dict(ckpt['model_state'])
    clf.eval()

    chunks = extract_sequences(video_path, ckpt['n_frames'])
    if not chunks: return {"error": "no sequence"}

    all_probs = []
    for seq in chunks:
        x = torch.tensor(seq, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            probs = torch.softmax(clf(x), dim=1).numpy()[0]
            all_probs.append(probs)
    
    avg_probs = np.mean(all_probs, axis=0)
    return {n: float(p) for n, p in zip(ckpt['class_names'], avg_probs)}


In [21]:
predict_clip("/kaggle/input/datasets/gallimus/cbms-training-data/spitting/VID-20260322-WA0088.mp4", "/kaggle/working/activity_classifier.pt")

{'littering': 2.362646611686614e-08,
 'normal': 1.1805630037997616e-06,
 'spitting': 0.9999987483024597}

## Cell 10 — Pipeline integration snippet
After downloading `activity_classifier.pt` and `class_labels.json`,
replace the MediaPipe heuristic in `pipeline_v2.ipynb` with this.

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Include Model Class
class PoseActivityClassifier(nn.Module):
    def __init__(self, input_size, hidden1, hidden2, num_classes, dropout):
        super().__init__()
        self.gru1 = nn.GRU(input_size, hidden1, batch_first=True, bidirectional=True)
        self.gru2 = nn.GRU(hidden1*2, hidden2, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden2*2, num_classes)
    def forward(self, x):
        out, _ = self.gru1(x)
        out, _ = self.gru2(out)
        out = torch.mean(out, dim=1)
        return self.classifier(self.dropout(out))

def load_activity_classifier(model_path: str):
    ckpt = torch.load(model_path, map_location="cuda" if torch.cuda.is_available() else "cpu")
    clf = PoseActivityClassifier(ckpt['features'], ckpt['hidden1'], ckpt['hidden2'], len(ckpt['class_names']), 0.0)
    clf.load_state_dict(ckpt['model_state'])
    clf.to("cuda" if torch.cuda.is_available() else "cpu").eval()
    return clf, ckpt['class_names']

def classify_crop_sequence(crops: list, classifier, class_names: list) -> tuple:
    if len(crops) < 2: return "normal", 0.0
    indices = np.linspace(0, len(crops)-1, 32, dtype=int) # N_FRAMES=32
    sequence = []
    for idx in indices:
        crop = crops[idx]
        try:
            rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            res = MP_POSE.process(rgb)
            if res.pose_landmarks:
                kp = res.pose_landmarks.landmark
                nose = kp[0]
                # MUST maintain Zero-Centering as trained
                vec = np.array([[lm.x-nose.x, lm.y-nose.y, lm.z] for lm in kp]).flatten()
            else: vec = np.zeros(99)
        except: vec = np.zeros(99)
        sequence.append(vec)
    
    x = torch.tensor(np.stack(sequence), dtype=torch.float32).unsqueeze(0).to(next(classifier.parameters()).device)
    with torch.no_grad():
        probs = F.softmax(classifier(x), dim=1).cpu().numpy()[0]
    
    best_idx = probs.argmax()
    return class_names[best_idx], float(probs[best_idx])
